# WAV → AudX (streaming) → Silero VAD → NVIDIA NeMo/Nemotron → Text

**Mục tiêu:** Replay tệp WAV theo từng chunk **80 ms** tại **16 kHz**, xử lý qua **checkpoint AudX đã train**, chạy **Silero VAD** để xác định utterance và **Nemotron 3.5 streaming ASR** để in `PARTIAL`/`FINAL`, lưu transcript ra TXT/CSV.

Pipeline dùng các module từ **source engine v0.3.3 (4)** do anh cung cấp, **không thay bằng `transcribe()` offline**. NeMo giữ encoder/RNNT cache qua các lệnh `STEP` trong cùng utterance; AudX giữ CNN context, STFT/iSTFT và LSTM state **xuyên suốt toàn WAV**, kể cả khi VAD kết thúc câu.

**Quy ước thời gian:** `AudxModel.step()` có lookahead tối đa 64 ms; bộ bọc thử nghiệm thêm **D = 80 + 64 + 100 = 244 ms = 3.904 samples**. Mỗi `push(chunk)` trả **đúng bằng số sample input**; audio trả ra là nguồn cũ hơn 244 ms. Khi `finish()` xuất nốt phần bị giữ, loại zero-padding đầu luồng để lưu `audx_aligned.wav`. `VAD` và `ASR` cùng nhìn luồng delayed; timestamp *source* được hiệu chỉnh `delivery_sample - D`.

**Yêu cầu trước khi chạy ASR thật:** WAV tiếng nói; ZIP engine `nemotron_translate_tts_engine_v0.3.3(4).zip` hoặc cây thư mục source; ZIP POC `AudX_Streaming_Telecom_POC.zip` (chứa `audx_weights_safe.pth`), hoặc đường dẫn checkpoint tương đương; checkpoint **`.nemo`** tương thích bản NeMo Speech engine đang dùng; `silero-vad` hoặc file JIT cục bộ. **Checkpoint `.nemo` không nằm trong notebook, cần cung cấp trên máy chạy.**

Không gọi API dịch/TTS; đầu ra là transcript ASR thuần.

## 1. Cấu hình đầu vào — sửa các đường dẫn trước khi Run All

In [ ]:
from pathlib import Path
import os

# Thư mục nguồn: Colab (/content), máy Linux, hoặc thư mục notebook hiện tại.
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
WAV_PATH = BASE / 'input.wav'                # WAV đầu vào: stereo hoặc sample rate khác vẫn đọc được
ENGINE_ROOT = BASE / 'engine'                # Nếu đã có src/nemotron_testbed, dùng trực tiếp
ENGINE_ZIP = BASE / 'nemotron_translate_tts_engine_v0.3.3(4).zip'
AUDX_POC_ZIP = BASE / 'AudX_Streaming_Telecom_POC.zip'
AUDX_CHECKPOINT = BASE / 'models' / 'audx' / 'audx_weights_safe.pth'
NEMO_MODEL_PATH = BASE / 'models' / 'nemotron-3.5-asr-streaming-0.6b' / 'nemotron-3.5-asr-streaming-0.6b.nemo'
SILERO_JIT_PATH = None   # ví dụ: str(BASE / 'models/silero/silero_vad.jit'), hoặc None để dùng package silero-vad

OUTPUT_DIR = BASE / 'audx_nemo_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
INPUT_CHUNK_MS = 80
ASR_LANGUAGE = 'vi-VN'    # 'en-US' nếu WAV tiếng Anh và checkpoint hỗ trợ
LOOKAHEAD_TOKENS = 6    # Nemotron supported: 0,1,3,6,13
LEFT_CONTEXT = 56
MODEL_CHUNK_MAPPING_MS = {0: 80, 1: 160, 3: 320, 6: 560, 13: 1120}
MODEL_CHUNK_MS = MODEL_CHUNK_MAPPING_MS[LOOKAHEAD_TOKENS]

AUDX_ENABLED = True
AUDX_DEVICE = 'cuda:0'       # Đổi thành 'cpu' nếu cần test AudX trên CPU
AUDX_AGC_TARGET_DBFS = -28.0
AUDX_AGC_MAX_GAIN_DB = 30.0
AUDX_HOP_MS = 80
AUDX_LOOKAHEAD_MS = 64
AUDX_COMPUTE_BUDGET_MS = 100
AUDX_DELAY_SAMPLES = SAMPLE_RATE * (AUDX_HOP_MS + AUDX_LOOKAHEAD_MS + AUDX_COMPUTE_BUDGET_MS) // 1000
assert AUDX_DELAY_SAMPLES == 3904

VAD_BACKEND = 'silero'   # 'energy' chỉ để smoke-test nếu chưa cài Silero, không đánh giá sản xuất
VAD_DEVICE = 'cpu'
VAD_THRESHOLD = 0.50
VAD_NEG_THRESHOLD = 0.35
VAD_MIN_SPEECH_MS = 250
VAD_END_SILENCE_MS = 250
VAD_PRE_ROLL_MS = 150

ASR_BACKEND = 'nemotron'  # 'mock' chỉ để kiểm tra wiring, KHÔNG sinh bản chép lời thật
ASR_DEVICE = 'cuda:0'
ASR_RESPONSE_TIMEOUT_S = 180
REALTIME_PACING = False # True = mô phỏng packet đến mỗi 80 ms; False = replay nhanh nhất có thể
PRINT_PARTIALS = True

print('WAV:', WAV_PATH)
print('ASR chunk:', MODEL_CHUNK_MS, 'ms; AudX delay:', AUDX_DELAY_SAMPLES / SAMPLE_RATE * 1000, 'ms')
print('Output:', OUTPUT_DIR)

## 2. Chuẩn bị môi trường và source

**Khuyến nghị:** chạy notebook bằng đúng venv/container GPU đã chạy được `NemotronRuntime` của engine. `NeMo Speech` ở đây phải có `nemo.collections.asr.models.rnnt_bpe_models_prompt.EncDecRNNTBPEModelWithPrompt`; cài một bản `nemo_toolkit` không tương thích có thể thiếu class đó.

Nếu dùng môi trường mới, cài phần nhẹ (`soundfile`, `scipy`, `silero-vad`, `prometheus-client`, `omegaconf`, `matplotlib`). PyTorch CUDA và NeMo nên cài đúng theo image/wheels đang chạy ASR. **Không gọi `torch.load(weights_only=False)` đối với checkpoint đầy đủ**; dùng checkpoint *weights-only* trong POC.

Lệnh cài tùy chọn (bỏ comment nếu thiếu package):

```python
# %pip install -q soundfile scipy silero-vad prometheus-client omegaconf matplotlib pandas
```

In [ ]:
import sys, zipfile

# Chỉ giải nén mã Python của engine trong ZIP người dùng đã cung cấp.
if not (ENGINE_ROOT / 'src/nemotron_testbed/runtime.py').is_file():
    if not ENGINE_ZIP.is_file():
        raise FileNotFoundError(f'Cần ENGINE_ROOT/src/... hoặc ENGINE_ZIP={ENGINE_ZIP}')
    with zipfile.ZipFile(ENGINE_ZIP) as z:
        source_files = [n for n in z.namelist()
                        if n.startswith('src/nemotron_testbed/') and n.endswith('.py') and '..' not in Path(n).parts]
        if not source_files:
            raise RuntimeError('Không tìm thấy source src/nemotron_testbed trong ZIP')
        for name in source_files:
            destination = ENGINE_ROOT / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(z.read(name))
    print(f'Extracted {len(source_files)} source files:', ENGINE_ROOT)

# Giải nén checkpoint weights-only (dùng checkpoint đã đóng gói từ model anh train).
if AUDX_ENABLED and not AUDX_CHECKPOINT.is_file():
    if not AUDX_POC_ZIP.is_file():
        raise FileNotFoundError('Thiếu checkpoint weights-only; cung cấp AUDX_CHECKPOINT hoặc AUDX_POC_ZIP')
    with zipfile.ZipFile(AUDX_POC_ZIP) as z:
        matches = [n for n in z.namelist() if n.endswith('/model/audx_weights_safe.pth')]
        if len(matches) != 1:
            raise RuntimeError(f'Không tìm thấy checkpoint weights-only trong {AUDX_POC_ZIP}')
        AUDX_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
        with z.open(matches[0]) as src, AUDX_CHECKPOINT.open('wb') as dst:
            import shutil
            shutil.copyfileobj(src, dst)

sys.path.insert(0, str(ENGINE_ROOT / 'src'))
import numpy as np
import soundfile as sf
import scipy.signal as sps
import torch
from IPython.display import display, Audio
from nemotron_testbed.audx_model import AudxModel
from nemotron_testbed.buffers import SampleBuffer
from nemotron_testbed.vad import VadEvent, build_vad_controller
from nemotron_testbed.runtime import NemotronRuntime
from nemotron_testbed.domain import StreamKey, RuntimeCommand, RuntimeCommandType, RuntimeResultType

print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('AudX checkpoint:', AUDX_CHECKPOINT if AUDX_ENABLED else 'bypass')

## 3. Đọc WAV → mono float32 → 16 kHz, hiển thị audio gốc

In [ ]:
from fractions import Fraction

if not WAV_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy WAV_PATH: {WAV_PATH}. Hãy upload WAV và sửa ô cấu hình.')

raw, original_sr = sf.read(str(WAV_PATH), dtype='float32', always_2d=True)
if raw.size == 0:
    raise ValueError('WAV rỗng')
if not np.isfinite(raw).all():
    raise ValueError('WAV có NaN/Inf')
mono = raw.mean(axis=1).astype(np.float32)
if original_sr != SAMPLE_RATE:
    ratio = Fraction(SAMPLE_RATE, int(original_sr))
    mono = sps.resample_poly(mono, ratio.numerator, ratio.denominator).astype(np.float32)
mono = np.clip(mono, -1., 1.)
N_INPUT = len(mono)
print(f'Original: {original_sr} Hz, {raw.shape[1]} channels; processed: {SAMPLE_RATE} Hz, mono, {N_INPUT/SAMPLE_RATE:.3f} s')
display(Audio(mono[:min(len(mono), 20*SAMPLE_RATE)], rate=SAMPLE_RATE))  # preview tối đa 20 s

## 4. AudX streaming — giữ LSTM state + cố định 244 ms delay

Sử dụng `AudxModel.step(state, data, final=False)` từ source engine, không gọi AudX riêng cho từng cửa sổ như offline. Bộ bọc dưới đây **chạy đồng bộ để kiểm tra thuật toán**, khác production ở chỗ production dùng queue/worker và có thể fallback về audio gốc nếu job trả muộn. 

Khi xử lý đầy đủ file, `push()` luôn trả đúng `len(input)`; `finish()` trả tail sao cho toàn bộ tín hiệu đã xử lý, kể cả padding, dài chính xác `N_INPUT + D`. Đường VAD/ASR được cấp theo *delivery timeline*, còn file `audx_aligned.wav` được bỏ D mẫu padding để nghe và so sánh với raw.

In [ ]:
class FixedDelayAudx:
    def __init__(self, checkpoint, device='cpu', delay_samples=3904,
                 agc_target_dbfs=-28., agc_max_gain_db=30.):
        self.model = AudxModel(checkpoint, device,
                               agc_target_dbfs=agc_target_dbfs,
                               agc_max_gain_db=agc_max_gain_db)
        self.state = self.model.new_state()
        self.delay = int(delay_samples)
        self.completed = SampleBuffer()   # đã enhance, chưa xuất về timeline nguồn
        self.in_count = 0
        self.source_released = 0
        self.out_count = 0
        self.model_ms = []

    def push(self, chunk):
        import time
        x = np.asarray(chunk, dtype=np.float32).reshape(-1)
        start = time.perf_counter()
        ready = self.model.step(self.state, x, final=False)
        self.model_ms.append((time.perf_counter() - start) * 1000)
        self.completed.append(ready)
        self.in_count += len(x)
        playable = max(0, self.in_count - self.delay) - self.source_released
        assert 0 <= playable <= len(x), 'Sai contract delay AudX'
        assert self.completed.samples >= playable, 'AudX chưa xuất đủ audio để đáp ứng delay cố định'
        audio = self.completed.pop_exact(playable)
        self.source_released += playable
        out = np.concatenate([np.zeros(len(x) - playable, np.float32), audio])
        assert len(out) == len(x)
        self.out_count += len(out)
        return out

    def finish(self):
        tail = self.model.step(self.state, np.empty(0, np.float32), final=True)
        self.completed.append(tail)
        missing_leading_pad = max(0, self.delay - self.in_count)
        remaining = self.in_count - self.source_released
        assert self.completed.samples == remaining, 'AudX xuất thiếu/dư sample cuối'
        data = self.completed.pop_all()
        out = np.concatenate([np.zeros(missing_leading_pad, np.float32), data])
        self.source_released += remaining
        self.out_count += len(out)
        assert self.source_released == self.in_count
        assert self.out_count == self.in_count + self.delay
        return out

class RawBypass:
    delay = 0
    model_ms = []
    def push(self, x):
        return np.asarray(x, dtype=np.float32).copy()
    def finish(self):
        return np.empty(0, dtype=np.float32)

print('AudX frame: STFT 512 / hop 256 @ 16 kHz; delay:', AUDX_DELAY_SAMPLES, 'samples')

## 5. NeMo/Nemotron streaming runtime — `START / STEP / FINALIZE`

Dùng chính `NemotronRuntime` của engine. Notebook chờ kết quả của mỗi lệnh trước khi gửi lệnh tiếp theo nên **không có rủi ro `FINALIZE` ưu tiên vượt qua `STEP` đang chờ**. Kết quả `PARTIAL` hiển thị khi utterance đang mở, kết quả `FINAL` ghi vào transcript.

`ASR_BACKEND='mock'` chỉ phục vụ **smoke test contract** khi chưa có model NVIDIA; tuyệt đối không xem đó là ASR thật.

In [ ]:
import queue, time
from types import SimpleNamespace

class NemoStreamingClient:
    def __init__(self):
        if not NEMO_MODEL_PATH.is_file():
            raise FileNotFoundError(f'Thiếu checkpoint NVIDIA .nemo: {NEMO_MODEL_PATH}')
        try:
            from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt
        except Exception as exc:
            raise ImportError('Cần NeMo Speech build đúng engine, chứa EncDecRNNTBPEModelWithPrompt; '
                              'hãy chạy notebook trong môi trường engine ASR.') from exc
        self.key = StreamKey('wav-replay', 'wav-session', 'mono')
        self.results = queue.Queue()
        self.runtime = NemotronRuntime(
            model_path=str(NEMO_MODEL_PATH), device=ASR_DEVICE, language=ASR_LANGUAGE,
            left_context=LEFT_CONTEXT, lookahead_tokens=LOOKAHEAD_TOKENS,
            queue_size=128, warmup_seconds=0., result_callback=self.results.put,
            beam_size=1, worker_id=0, suppress_model_init_output=True,
        )
        self.runtime.start(timeout=600.)

    def _send(self, command, expected_type):
        self.runtime.submit(command, timeout=ASR_RESPONSE_TIMEOUT_S)
        result = self.results.get(timeout=ASR_RESPONSE_TIMEOUT_S)
        if result.type is RuntimeResultType.ERROR:
            raise RuntimeError(f'NeMo {command.type.name} failed:\n{result.error}')
        if result.type is not expected_type or result.utterance_id != command.utterance_id:
            raise RuntimeError(f'ASR event unexpected: {result.type}, utterance={result.utterance_id}')
        return result

    def start(self, utterance_id):
        return self._send(RuntimeCommand(type=RuntimeCommandType.START,
                                         key=self.key, utterance_id=utterance_id,
                                         language=ASR_LANGUAGE), RuntimeResultType.STARTED)

    def step(self, utterance_id, samples):
        return self._send(RuntimeCommand(type=RuntimeCommandType.STEP,
                                         key=self.key, utterance_id=utterance_id,
                                         audio=samples), RuntimeResultType.PARTIAL)

    def finalize(self, utterance_id, samples):
        return self._send(RuntimeCommand(type=RuntimeCommandType.FINALIZE,
                                         key=self.key, utterance_id=utterance_id,
                                         audio=samples), RuntimeResultType.FINAL)

    def close(self):
        self.runtime.shutdown(timeout=15.)

class MockASR:
    """CHỈ kiểm tra pipeline, không hề nhận dạng giọng nói."""
    def start(self, utterance_id):
        return SimpleNamespace(text='', latency_ms=0)
    def step(self, utterance_id, samples):
        return SimpleNamespace(text='[MOCK PARTIAL: KHÔNG PHẢI ASR]', latency_ms=0)
    def finalize(self, utterance_id, samples):
        return SimpleNamespace(text='[MOCK FINAL: KHÔNG PHẢI BẢN CHÉP LỜI]', latency_ms=0)
    def close(self):
        pass

if ASR_BACKEND == 'nemotron':
    ASR = NemoStreamingClient()
elif ASR_BACKEND == 'mock':
    print('WARNING: MOCK MODE, transcript sinh ra KHÔNG chứa nội dung tiếng nói thật.')
    ASR = MockASR()
else:
    raise ValueError('ASR_BACKEND phải là nemotron hoặc mock')

## 6. Điều phối VAD/ASR với pre-roll và timeline

- **VAD:** đúng module `VadController` và `SileroSpeechProbability` từ engine; Silero xử lý cửa sổ 512 samples/32 ms bên trong, dù bên ngoài nhận chunk 80 ms.
- **START:** nối pre-roll để không mất âm đầu; tạo state NeMo utterance mới.
- **SPEECH:** gửi audio theo bước `MODEL_CHUNK_MS`; `PARTIAL` không được nối trực tiếp vào transcript final.
- **END:** gửi phần audio còn lại và gọi `FINALIZE`; `end_tail_ms` của VAD được giữ lại cho pre-roll kế tiếp.
- **EOF:** nếu file kết thúc khi VAD còn đang active, `FINALIZE` bắt buộc để không mất câu cuối.
- **Timestamp:** ước lượng thời gian dựa trên media sample clock (không phải accurate word timestamps), trừ 244 ms delay từ delivery timeline.

In [ ]:
from collections import deque
from dataclasses import dataclass

class VadToNemoPipeline:
    def __init__(self, asr, enhancer_delay_samples, chunk_ms):
        self.asr = asr
        self.delay = int(enhancer_delay_samples)
        self.chunk_ms = int(chunk_ms)
        self.vad = build_vad_controller(
            backend=VAD_BACKEND, enabled=True, threshold=VAD_THRESHOLD,
            neg_threshold=VAD_NEG_THRESHOLD, energy_db_threshold=-42.0,
            min_speech_ms=VAD_MIN_SPEECH_MS, end_silence_ms=VAD_END_SILENCE_MS,
            silero_model_path=str(SILERO_JIT_PATH) if SILERO_JIT_PATH else None,
            device=VAD_DEVICE,
        )
        preroll_samples = int((VAD_PRE_ROLL_MS + VAD_MIN_SPEECH_MS + 64 + INPUT_CHUNK_MS)
                               * SAMPLE_RATE / 1000)
        self.pre_roll = SampleBuffer(max_samples=preroll_samples)
        self.current_id = None
        self.utterance_counter = 0
        self.utterance_start_delivery = 0
        self.pending_asr = SampleBuffer()
        self.delivered_pos = 0
        self.partial_previous = ''
        self.transcripts = []
        self.vad_events = []
        self.asr_step_ms = []
        self.audio_to_asr = 0
        self.step_count = 0

    def _feed_asr(self, audio):
        self.pending_asr.append(audio)
        model_chunk_samples = MODEL_CHUNK_MS * SAMPLE_RATE // 1000
        while self.pending_asr.samples >= model_chunk_samples:
            block = self.pending_asr.pop_exact(model_chunk_samples)
            t0 = time.perf_counter()
            res = self.asr.step(self.current_id, block)
            self.asr_step_ms.append((time.perf_counter() - t0) * 1000)
            self.audio_to_asr += len(block)
            self.step_count += 1
            if PRINT_PARTIALS and res.text and res.text != self.partial_previous:
                print(f'[PARTIAL u{self.current_id:03d}] {res.text}')
            self.partial_previous = res.text

    def _start_utt(self):
        self.utterance_counter += 1
        self.current_id = self.utterance_counter
        self.utterance_start_delivery = self.delivered_pos - self.pre_roll.samples
        self.asr.start(self.current_id)
        self._feed_asr(self.pre_roll.pop_all())
        self.partial_previous = ''

    def _finish_utt(self, delivery_end, reason):
        if self.current_id is None:
            return
        remainder = self.pending_asr.pop_all()
        self.audio_to_asr += len(remainder)
        t0 = time.perf_counter()
        res = self.asr.finalize(self.current_id, remainder)
        finalize_ms = (time.perf_counter()-t0)*1000
        source_begin_ms = max(0, self.utterance_start_delivery - self.delay)*1000.0/SAMPLE_RATE
        source_end_ms = max(0, delivery_end - self.delay)*1000.0/SAMPLE_RATE
        row = {
            'utterance_id': self.current_id,
            'start_source_s_approx': round(source_begin_ms/1000., 3),
            'end_source_s_approx': round(source_end_ms/1000., 3),
            'duration_asr_s': round(self.audio_to_asr / SAMPLE_RATE, 3),
            'text': res.text,
            'reason': reason,
            'asr_finalize_ms': round(finalize_ms, 2),
            'asr_step_count': self.step_count,
        }
        self.transcripts.append(row)
        print(f'[FINAL u{self.current_id:03d}, {reason}] {res.text}')
        self.current_id = None
        self.pending_asr.clear()
        self.partial_previous = ''
        self.audio_to_asr = 0
        self.step_count = 0

    def ingest(self, audio):
        block = np.asarray(audio, dtype=np.float32).reshape(-1)
        if not len(block):
            return
        previous_pos = self.delivered_pos
        self.delivered_pos += len(block)
        event = self.vad.process(block, len(block) * 1000.0 / SAMPLE_RATE)
        score = float(getattr(self.vad.probability, '_last_score', np.nan))
        self.vad_events.append({
            'delivery_s': self.delivered_pos / SAMPLE_RATE,
            'source_s': max(0, self.delivered_pos - self.delay) / SAMPLE_RATE,
            'event': event.name, 'score': score,
        })
        if self.current_id is None:
            self.pre_roll.append(block)
            if event is VadEvent.START:
                self._start_utt()
            return
        if event is VadEvent.END:
            tail_samples = max(0, min(len(block), round(self.vad.end_tail_ms * SAMPLE_RATE / 1000.0)))
            body_end = len(block) - tail_samples
            self._feed_asr(block[:body_end])
            self._finish_utt(previous_pos + body_end, reason='VAD_END')
            self.pre_roll.clear()
            if tail_samples:
                self.pre_roll.append(block[body_end:])
        else:
            self._feed_asr(block)

    def finish(self):
        if self.current_id is not None:
            self._finish_utt(self.delivered_pos, reason='EOF_FLUSH')
        if self.utterance_counter == 0:
            print('Không phát hiện speech theo VAD config. Hãy nghe audio enhanced và kiểm tra threshold.')

print('VAD:', VAD_BACKEND, '| endpoint:', VAD_END_SILENCE_MS, 'ms | ASR:', ASR_BACKEND)

## 7. Chạy toàn bộ WAV theo chu kỳ chunk 80 ms

Chạy một media stream liên tục qua AudX; VAD tạo một hoặc nhiều utterance và NeMo sử dụng cùng model nhưng state mới theo utterance. **Không pad thêm audio vào đầu WAV**: pipeline tạo D mẫu trễ theo đúng contract.

Ở mode `REALTIME_PACING=False`, đây là **replay offline theo chunk** (thứ tự streaming đúng nhưng không mô phỏng tốc độ mạng). Bật `True` để phát chunk xấp xỉ chu kỳ 80 ms và kiểm tra backlog/timing; điều đó vẫn không phải benchmark concurrency production.

In [ ]:
import time

if AUDX_ENABLED:
    enhancer = FixedDelayAudx(AUDX_CHECKPOINT, device=AUDX_DEVICE,
                              delay_samples=AUDX_DELAY_SAMPLES,
                              agc_target_dbfs=AUDX_AGC_TARGET_DBFS,
                              agc_max_gain_db=AUDX_AGC_MAX_GAIN_DB)
else:
    enhancer = RawBypass()

pipeline = VadToNemoPipeline(ASR, enhancer.delay, INPUT_CHUNK_MS)
input_chunk_samples = INPUT_CHUNK_MS * SAMPLE_RATE // 1000
wave_blocks = []
start_wall = time.perf_counter()
try:
    for idx, i in enumerate(range(0, N_INPUT, input_chunk_samples)):
        input_block = mono[i:i + input_chunk_samples]
        processed = enhancer.push(input_block)
        assert len(processed) == len(input_block)
        wave_blocks.append(processed)
        pipeline.ingest(processed)
        if REALTIME_PACING:
            target_sec = (i + len(input_block)) / SAMPLE_RATE
            to_sleep = target_sec - (time.perf_counter() - start_wall)
            if to_sleep > 0:
                time.sleep(to_sleep)

    final_audio = enhancer.finish()
    wave_blocks.append(final_audio)
    for i in range(0, len(final_audio), input_chunk_samples):
        pipeline.ingest(final_audio[i:i + input_chunk_samples])
    pipeline.finish()
finally:
    ASR.close()

wave_delayed = np.concatenate(wave_blocks) if wave_blocks else np.empty(0, dtype=np.float32)
assert len(wave_delayed) == N_INPUT + enhancer.delay, (len(wave_delayed), N_INPUT, enhancer.delay)
wave_aligned = wave_delayed[enhancer.delay:enhancer.delay+N_INPUT]
assert len(wave_aligned) == N_INPUT
sf.write(OUTPUT_DIR / 'audx_aligned.wav', wave_aligned, SAMPLE_RATE)
sf.write(OUTPUT_DIR / 'audx_with_delay.wav', wave_delayed, SAMPLE_RATE)
wall_sec = time.perf_counter() - start_wall
print(f'Completed: {N_INPUT / SAMPLE_RATE:.2f} audio sec; {wall_sec:.2f} wall sec; '
      f'audio frames: {len(wave_blocks)}, utterances: {len(pipeline.transcripts)}')
print('Enhanced audio saved:', OUTPUT_DIR / 'audx_aligned.wav')

## 8. Transcript, thống kê latency và playback audio kết quả

In [ ]:
import pandas as pd
from IPython.display import display

transcript_df = pd.DataFrame(pipeline.transcripts)
if len(transcript_df):
    display(transcript_df)
    output_txt = OUTPUT_DIR / 'transcript.txt'
    lines = [f"[{r['start_source_s_approx']:.3f} - {r['end_source_s_approx']:.3f}s] {r['text']}"
             for r in pipeline.transcripts]
    output_txt.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    transcript_df.to_csv(OUTPUT_DIR / 'transcript.csv', index=False, encoding='utf-8-sig')
    print('\n'.join(lines))
    print('TXT:', output_txt)
else:
    print('Không có utterance để ghi. Kiểm tra VAD/ASR config.')

print('AudX inference P95 (ms/chunk):',
      round(float(np.percentile(enhancer.model_ms, 95)), 2) if enhancer.model_ms else 'n/a')
print('NeMo STEP P95 (ms/STEP):',
      round(float(np.percentile(pipeline.asr_step_ms, 95)), 2) if pipeline.asr_step_ms else 'n/a')
print('Pipeline wall / audio duration:', round(wall_sec/(N_INPUT/SAMPLE_RATE), 3))
display(Audio(wave_aligned[:min(len(wave_aligned), 20*SAMPLE_RATE)], rate=SAMPLE_RATE))  # preview tối đa 20 s

## 9. Biểu đồ VAD và waveform để debug

In [ ]:
import matplotlib.pyplot as plt

plot_end = min(len(mono), 60 * SAMPLE_RATE)
plot_stride = max(1, plot_end // 8000)
t = np.arange(0, plot_end, plot_stride) / SAMPLE_RATE
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, mono[:plot_end:plot_stride], linewidth=0.6, alpha=0.60, label='Raw WAV')
ax.plot(t, wave_aligned[:plot_end:plot_stride], linewidth=0.6, alpha=0.65, label='AudX aligned')
for u in pipeline.transcripts:
    ax.axvspan(u['start_source_s_approx'], u['end_source_s_approx'], alpha=0.10)
ax.set_xlabel('Source audio timeline (seconds)')
ax.set_ylabel('Amplitude')
ax.set_title('WAV vs AudX enhancement / VAD segments (first 60 seconds max)')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

if pipeline.vad_events:
    event_x = [e['source_s'] for e in pipeline.vad_events]
    event_y = [e['score'] for e in pipeline.vad_events]
    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.plot(event_x, event_y, marker='.', linewidth=1, label='Silero score (latest window)')
    ax.axhline(VAD_THRESHOLD, linestyle='--', label='Speech threshold')
    ax.axhline(VAD_NEG_THRESHOLD, linestyle=':', label='Negative threshold')
    for ev in pipeline.vad_events:
        if ev['event'] in ('START','END'):
            ax.axvline(ev['source_s'], linestyle=':', alpha=0.7)
    ax.set(xlabel='Source timeline (seconds)', ylabel='VAD probability', ylim=(-0.05,1.05))
    ax.set_title('VAD activity / START-END markers')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.25)
    plt.show()

## 10. Điều kiện chấp nhận và lưu ý vận hành

**Kết quả cần kiểm tra:**

1. `len(AudX.push(chunk)) == len(chunk)` với mọi chunk, và `len(delayed_output) == len(WAV_16k) + 3904` sau flush; WAV aligned đúng bằng độ dài input.
2. VAD thấy **streaming output đã delay**; timestamp source trong bảng được bù 244 ms, nhưng thời gian bắt đầu chỉ là **ước lượng có pre-roll**, không phải word alignment.
3. `PARTIAL` có thể được sửa ở STEP tiếp theo; chỉ `FINAL` mới được lưu thành text của utterance. Nếu VAD báo END sớm, transcript có thể mất context.
4. `AudxModel.step` **không reset khi VAD END**. `NemotronRuntime` **reset state khi bắt đầu utterance mới**, đúng với pipeline engine.
5. **Trường hợp im lặng không có RTP (DTX)** không được mô phỏng khi đọc WAV; cần testcase riêng ở môi trường RTP thực.
6. Notebook dùng AudX synchronous, không mô phỏng queue wait/bypass worker của engine. P95 đo ở đây chỉ tham khảo; muốn benchmark thực tế phải đo nhiều concurrent calls trên GPU.
7. **Nếu tiếng Việt không ra đúng text**, kiểm tra checkpoint `.nemo`, `language='vi-VN'`, WAV gốc, WAV AudX aligned, VAD threshold và model chunk/lookahead. Để A/B, chạy lại với `AUDX_ENABLED=False`.

**Thiếu NeMo checkpoint trên môi trường notebook:** chuyển `ASR_BACKEND='mock'` *chỉ để thử plumbing*. Kết quả mock không phải bản chép lời. Để có text thật bắt buộc chạy `ASR_BACKEND='nemotron'` với checkpoint/NeMo build phù hợp.